In [1]:
import torch
import open_clip
from PIL import Image
import chromadb
import os
from pathlib import Path

In [2]:

device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess, tokenizer = open_clip.create_model_and_transforms(
    "ViT-H-14",
    pretrained="laion2b_s32b_b79k"
)

model = model.to(device)
model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-31): 32 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1280, out_features=1280, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1280, out_features=5120, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=5120, out_features=1280, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1280,), eps=1e-05, elementwi

In [3]:
!nvidia-smi

Wed Feb 18 17:34:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.83                 Driver Version: 581.83         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650      WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   49C    P0             14W /   50W |    3873MiB /   4096MiB |     34%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
def generate_image_embedding(image_path):
    image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)

    with torch.no_grad():
        features = model.encode_image(image)

    features = features / features.norm(dim=-1, keepdim=True)

    return features.cpu().numpy()[0]

In [5]:
client = chromadb.PersistentClient(path='./chroma_db_pharaohs')

collection = client.create_collection(name="pharaohs_video_images")

In [6]:
image_folder = Path("Final_video_pharaohs")

image_files = list(image_folder.rglob("*.jpg")) + list(image_folder.rglob("*.png")) + list(image_folder.rglob("*.jpeg"))
image_id = 0
for img_path in image_files:
    emb = generate_image_embedding(str(img_path))
    
    pharaoh_name = img_path.parent.name
    file_name = img_path.stem

    collection.add(
        ids=[str(image_id)],  
        embeddings=[emb],
        metadatas=[{"pharaoh_name": pharaoh_name,
                    "image_description":file_name}],
        documents=[str(img_path)]
    )
    image_id+=1

In [ ]:
print(collection.count())

959


In [11]:
results = collection.get(limit=11)
for meta, doc in zip(results["metadatas"], results["documents"]):
    print(f"{doc} → {meta}")

Final_video_pharaohs\Ahmose I\Ahmose I cartouche relief.jpg → {'pharaoh_name': 'Ahmose I', 'image_description': 'Ahmose I cartouche relief'}
Final_video_pharaohs\Ahmose I\Ahmose I cartouche.jpg → {'image_description': 'Ahmose I cartouche', 'pharaoh_name': 'Ahmose I'}
Final_video_pharaohs\Ahmose I\Ahmose I Cult Stele.jpg → {'pharaoh_name': 'Ahmose I', 'image_description': 'Ahmose I Cult Stele'}
Final_video_pharaohs\Ahmose I\Ahmose I pyramid plan.jpg → {'image_description': 'Ahmose I pyramid plan', 'pharaoh_name': 'Ahmose I'}
Final_video_pharaohs\Ahmose I\Ahmose I relief.jpg → {'image_description': 'Ahmose I relief', 'pharaoh_name': 'Ahmose I'}
Final_video_pharaohs\Ahmose I\Ahmose I slaying a Hyksos.jpg → {'image_description': 'Ahmose I slaying a Hyksos', 'pharaoh_name': 'Ahmose I'}
Final_video_pharaohs\Ahmose I\Ahmose I stele to grandmother Queen Tetisheri.jpg → {'image_description': 'Ahmose I stele to grandmother Queen Tetisheri', 'pharaoh_name': 'Ahmose I'}
Final_video_pharaohs\Ahmose

In [ ]:
results = collection.get(include=["embeddings", "metadatas", "documents"], limit=2)
embeddings = results["embeddings"]
print(len(embeddings[0]))

1024
